In [ ]:
# =============================================================================
# jobs/18_fgs_changes.py
# What changed in the forecast since the previous Flood Guidance Statement?
#
# A statement that says "Medium" tells you little. A statement that says
# "Medium, up from Low yesterday" tells you what to act on. This notebook
# compares the latest complete statement with the one before it, per
# constituency, forecast day and flood source, and records every difference.
#
# What this script does:
#   1. Takes the two newest statements whose intersections are complete for
#      FGS_GEOMETRY_SOURCE (the same rule as 14).
#   2. Works out the forecast days both statements cover. Only those days are
#      compared. A day that has rolled off the old statement, or a day newly
#      added to the new one, is not a change, so it is left out.
#   3. Takes the worst risk per constituency, day and flood source in each
#      statement (polygons covering less than FGS_CONTACT_MIN_PCT of the
#      constituency are ignored, as in 14).
#   4. Writes one row per difference to fgs_constituency_changes, replacing any
#      rows already held for this statement, so a re-run never duplicates.
#
# change_type: new (nothing before, a risk now), escalated (rank up),
#              de_escalated (rank down, still a risk), cleared (a risk before,
#              none now). Unchanged cells are not stored.
# Rank: 1 Very Low, 2 Low, 3 Medium, 4 High. 0 means no risk area covers the
# constituency for that day and source.
#
# NO SPATIAL WORK HERE. Plain Spark SQL over fgs_constituency_intersections.
#
# Exit values: "success" (rows written, possibly none), "no_previous" (only one
# complete statement exists, so there is nothing to compare).
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from datetime import date, timedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT,
    TBL_FGS_CHANGES, FGS_CONTACT_MIN_PCT, FGS_GEOMETRY_SOURCE
)
from utils.helpers import (
    get_spark, utc_now, table_exists, source_predicate,
    latest_complete_statements, replace_statement_rows, risk_rank_sql
)


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_FGS_CONST_INTERSECT):
    if not table_exists(spark, table):
        raise Exception(f"{table} does not exist. Run 01/02 and 07_compute_intersections first.")

CHANGES_SCHEMA = StructType([
    StructField("statement_id",          IntegerType(), True),
    StructField("previous_statement_id", IntegerType(), True),
    StructField("geometry_source",       StringType(),  True),
    StructField("issued_at",             StringType(),  True),
    StructField("previous_issued_at",    StringType(),  True),
    StructField("forecast_date",         StringType(),  True),
    StructField("day_index",             IntegerType(), True),
    StructField("source",                StringType(),  True),
    StructField("constituency_id",       StringType(),  True),
    StructField("constituency_name",     StringType(),  True),
    StructField("previous_rank",         IntegerType(), True),
    StructField("new_rank",              IntegerType(), True),
    StructField("rank_change",           IntegerType(), True),
    StructField("change_type",           StringType(),  True),
    StructField("computed_at",           StringType(),  True),
])


# Create the table (empty) on first run. Without this it would not exist until
# the first change is written, and anything that queries it (19, the dashboard)
# would fail on a day with nothing to report.
if not table_exists(spark, TBL_FGS_CHANGES):
    (
        spark.createDataFrame([], CHANGES_SCHEMA).write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_FGS_CHANGES)
    )
    print(f"Created {TBL_FGS_CHANGES}.")

complete = latest_complete_statements(
    spark, TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, n=2
)
if not complete:
    raise Exception(
        f"No statement has up-to-date intersections for geometry source "
        f"'{FGS_GEOMETRY_SOURCE}'. Run 07_compute_intersections first."
    )
if len(complete) < 2:
    print("Only one complete statement exists, so there is nothing to compare.")
    dbutils.notebook.exit("no_previous")

new, prev = complete[0], complete[1]
print(f"FGS_GEOMETRY_SOURCE = {FGS_GEOMETRY_SOURCE}")
print(f"Comparing statement {new['statement_id']} (issued {new['issued_at']}) "
      f"with {prev['statement_id']} (issued {prev['issued_at']}).")


In [ ]:
# =============================================================================
# WHICH FORECAST DAYS DO BOTH STATEMENTS COVER?
# A statement covers five days, day 1 to day 5. Day 1's date is read from the
# statement's own rows (forecast_date minus day_index) so nothing depends on a
# time zone assumption. A statement with no rows at all (a quiet one) falls
# back to the date it was issued.
# =============================================================================

i_in_source = source_predicate(spark, TBL_FGS_CONST_INTERSECT, FGS_GEOMETRY_SOURCE, alias="i")

day1_rows = spark.sql(f"""
    SELECT i.statement_id,
           MIN(date_sub(to_date(i.forecast_date), CAST(i.day_index AS INT) - 1)) AS day1_date
    FROM {TBL_FGS_CONST_INTERSECT} i
    WHERE i.statement_id IN ({new['statement_id']}, {prev['statement_id']})
      AND {i_in_source}
      AND i.forecast_date IS NOT NULL AND i.day_index IS NOT NULL
    GROUP BY i.statement_id
""").collect()
day1 = {int(r["statement_id"]): r["day1_date"] for r in day1_rows}

issued = spark.sql(f"""
    SELECT statement_id, to_date(issued_at) AS issued_date
    FROM {TBL_STATEMENTS}
    WHERE statement_id IN ({new['statement_id']}, {prev['statement_id']})
""").collect()
issued_date = {int(r["statement_id"]): r["issued_date"] for r in issued}

d1_new  = day1.get(int(new["statement_id"]))  or issued_date[int(new["statement_id"])]
d1_prev = day1.get(int(prev["statement_id"])) or issued_date[int(prev["statement_id"])]

# Statement days run d1 .. d1 + 4. Overlap: later start to earlier end.
start = max(d1_new, d1_prev)
end   = min(d1_new, d1_prev) + timedelta(days=4)
print(f"Day 1 of the new statement is {d1_new}; of the previous one {d1_prev}.")
print(f"Days compared: {start} to {end}.")

if start > end:
    print("The two statements share no forecast days (a gap in the history). Nothing to compare.")
    replace_statement_rows(spark, TBL_FGS_CHANGES, [new["statement_id"]], [], CHANGES_SCHEMA,
                           "geometry_source", FGS_GEOMETRY_SOURCE, "risk_areas")
    dbutils.notebook.exit("success")

# Self-check: day 1 is normally the day of issue. A different offset is not an
# error, but it is worth knowing about.
offset = (d1_new - issued_date[int(new["statement_id"])]).days
if offset != 0:
    print(f"NOTE: day 1 of the new statement is {offset} day(s) away from its issue date.")


In [ ]:
# =============================================================================
# COMPARE
# =============================================================================

def worst_by_cell(statement_id):
    """Worst risk rank per constituency, day and flood source for one statement."""
    return f"""
        SELECT i.constituency_id,
               MAX(i.constituency_name)                          AS constituency_name,
               i.forecast_date,
               i.source,
               MAX({risk_rank_sql('i.risk_level')})              AS risk_rank
        FROM {TBL_FGS_CONST_INTERSECT} i
        WHERE i.statement_id = {int(statement_id)}
          AND {i_in_source}
          AND (i.intersection_pct IS NULL OR i.intersection_pct >= {FGS_CONTACT_MIN_PCT})
          AND to_date(i.forecast_date) BETWEEN DATE '{start}' AND DATE '{end}'
        GROUP BY i.constituency_id, i.forecast_date, i.source
    """

query = f"""
SELECT
    COALESCE(n.constituency_id,   p.constituency_id)    AS constituency_id,
    COALESCE(n.constituency_name, p.constituency_name)  AS constituency_name,
    COALESCE(n.forecast_date,     p.forecast_date)      AS forecast_date,
    COALESCE(n.source,            p.source)             AS source,
    COALESCE(p.risk_rank, 0)                            AS previous_rank,
    COALESCE(n.risk_rank, 0)                            AS new_rank
FROM ({worst_by_cell(new['statement_id'])})  n
FULL OUTER JOIN ({worst_by_cell(prev['statement_id'])}) p
  ON  n.constituency_id = p.constituency_id
  AND n.forecast_date   = p.forecast_date
  AND n.source          = p.source
"""

changes = spark.sql(f"""
    SELECT *,
           CASE
               WHEN previous_rank = 0 AND new_rank > 0 THEN 'new'
               WHEN previous_rank > 0 AND new_rank = 0 THEN 'cleared'
               WHEN new_rank > previous_rank           THEN 'escalated'
               ELSE 'de_escalated'
           END AS change_type
    FROM ({query})
    WHERE previous_rank <> new_rank
""").collect()

counts = {}
for r in changes:
    counts[r["change_type"]] = counts.get(r["change_type"], 0) + 1
print(f"Changes: {len(changes)}   {counts}")

rows = []
for r in changes:
    rows.append({
        "statement_id":          int(new["statement_id"]),
        "previous_statement_id": int(prev["statement_id"]),
        "geometry_source":       FGS_GEOMETRY_SOURCE,
        "issued_at":             new["issued_at"],
        "previous_issued_at":    prev["issued_at"],
        "forecast_date":         r["forecast_date"],
        "day_index":             (date.fromisoformat(str(r["forecast_date"])[:10]) - d1_new).days + 1,
        "source":                r["source"],
        "constituency_id":       r["constituency_id"],
        "constituency_name":     r["constituency_name"],
        "previous_rank":         int(r["previous_rank"]),
        "new_rank":              int(r["new_rank"]),
        "rank_change":           int(r["new_rank"]) - int(r["previous_rank"]),
        "change_type":           r["change_type"],
        "computed_at":           now_iso,
    })


In [ ]:
# =============================================================================
# WRITE TO DELTA
# Rows for this statement and geometry source are replaced, so re-running never
# duplicates. Earlier statements' rows stay: the table is the history of changes.
# =============================================================================


written = replace_statement_rows(
    spark, TBL_FGS_CHANGES, [new["statement_id"]], rows, CHANGES_SCHEMA,
    source_column="geometry_source", source_value=FGS_GEOMETRY_SOURCE,
    legacy_source="risk_areas",
)

if table_exists(spark, TBL_FGS_CHANGES):
    spark.sql(f"""
        COMMENT ON TABLE {TBL_FGS_CHANGES} IS
        'Differences between each Flood Guidance Statement and the one before it, per
         constituency, forecast day and flood source. Written by 18_fgs_changes for the
         latest complete statement on every poll. Only forecast days that both
         statements cover are compared. previous_rank and new_rank: 1 Very Low, 2 Low,
         3 Medium, 4 High, 0 none. change_type: new, escalated, de_escalated, cleared.
         Unchanged cells are not stored.'
    """)

print(f"Wrote {written} row(s) for statement {new['statement_id']} to {TBL_FGS_CHANGES}.")
dbutils.notebook.exit("success")
